# Breast Cancer Dual & Triple AND Co-expression Target Analysis

**Dataset:** TCGA Breast Invasive Carcinoma (`denseDataOnlyDownload-2.tsv`)

**Logic:** AND — all genes in a combination must be simultaneously expressed above the tumor-derived median threshold.

**Approach:** Distribution analysis → SMOTE-balanced classification → AND co-expression scoring → model comparison (RF, DT, LR)

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from itertools import combinations
from scipy import stats
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

## 2. Load, clean & map gene names

In [ ]:
df = pd.read_csv('denseDataOnlyDownload-2.tsv', sep='\t')

# Map ENSEMBL IDs to gene symbols
ensembl_map = {
    'ENSG00000082175.15': 'PGR',
    'ENSG00000141736.14': 'ERBB2',
    'ENSG00000087586.18': 'SLAMF7',
    'ENSG00000171862.11': 'PTEN',
    'ENSG00000146648.19': 'EGFR',
    'ENSG00000103855.18': 'CD276',
    'ENSG00000090339.9':  'ICAM1'
}
df = df.rename(columns=ensembl_map)
df = df.rename(columns={
    'sample_type.samples': 'sample_type',
    'primary_diagnosis.diagnoses': 'primary_diagnosis'
})
GENES = list(ensembl_map.values())

# Remove nulls
before = len(df)
df = df.dropna(subset=GENES + ['sample_type'])
print(f'Removed {before - len(df)} rows with nulls -> {len(df)} samples remaining')
print(f'\nSample types:')
print(df['sample_type'].value_counts())

## 3. Split tumor vs normal

In [ ]:
tumor  = df[df['sample_type'].isin(['Primary Tumor', 'Metastatic'])].copy()
normal = df[df['sample_type'] == 'Solid Tissue Normal'].copy()
combined = pd.concat([tumor, normal], ignore_index=True)
combined['label'] = (combined['sample_type'].isin(['Primary Tumor', 'Metastatic'])).astype(int)

print(f'Tumor samples:  {len(tumor):,}')
print(f'Normal samples: {len(normal):,}')
print(f'Class imbalance ratio: {len(tumor)/len(normal):.1f}:1')

## 4. Distributional analysis — tumor vs normal

In [ ]:
print(f"{'Gene':<10} {'T_mean':>8} {'T_sd':>7} {'T_skew':>8} {'T_kurt':>8} | {'N_mean':>8} {'N_sd':>7} | {'Fold':>6} {'p-value':>12} {'Sig':>5}")
print('-' * 85)
for g in GENES:
    t = tumor[g].dropna().values
    n = normal[g].dropna().values
    _, p = stats.mannwhitneyu(t, n, alternative='two-sided')
    fold = t.mean() / n.mean()
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    direction = 'T up' if fold >= 1 else 'N up'
    print(f"{g:<10} {t.mean():>8.3f} {t.std():>7.3f} {stats.skew(t):>8.3f} {stats.kurtosis(t):>8.3f} | {n.mean():>8.3f} {n.std():>7.3f} | {fold:>6.3f} {p:>12.2e} {sig:>5}  {direction}")

## 5. Histograms — all 7 genes

In [ ]:
tumor_medians = {g: tumor[g].median() for g in GENES}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, g in enumerate(GENES):
    ax = axes[i]
    t = tumor[g].dropna().values
    n = normal[g].dropna().values
    bins = np.linspace(np.concatenate([t, n]).min(), np.concatenate([t, n]).max(), 35)

    ax.hist(t, bins=bins, alpha=0.60, color='#D85A30', label='Tumor',  density=True)
    ax.hist(n, bins=bins, alpha=0.70, color='#378ADD', label='Normal', density=True)
    ax.axvline(tumor_medians[g], color='black', linestyle='--', linewidth=1.3, label='Tumor median')

    _, p = stats.mannwhitneyu(t, n, alternative='two-sided')
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    fold = t.mean() / n.mean()
    direction = 'T up' if fold >= 1 else 'N up'
    sk_t, sk_n = stats.skew(t), stats.skew(n)
    ku_t = stats.kurtosis(t)

    ax.set_title(f'{g}  ({sig}, {fold:.2f}x {direction})', fontsize=9, fontweight='bold')
    ax.set_xlabel('Expression', fontsize=8)
    ax.set_ylabel('Density', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.text(0.97, 0.95,
            f'T skew: {sk_t:.2f}\nN skew: {sk_n:.2f}\nT kurt: {ku_t:.2f}',
            transform=ax.transAxes, fontsize=7, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
    if i == 0:
        ax.legend(fontsize=7)

axes[-1].set_visible(False)
plt.suptitle('Gene Expression Distributions: Tumor vs Normal Breast Tissue\n(dashed = tumor median threshold; density-normalized)', fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig1_histograms.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as fig1_histograms.png')

## 6. Boxplots — tumor vs normal per gene

In [ ]:
fig, axes = plt.subplots(1, len(GENES), figsize=(18, 5))
for i, g in enumerate(GENES):
    ax = axes[i]
    data = [tumor[g].values, normal[g].values]
    bp = ax.boxplot(data, labels=['Tumor', 'Normal'], patch_artist=True, widths=0.5,
                    medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor('#D85A30'); bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor('#378ADD'); bp['boxes'][1].set_alpha(0.7)
    ax.set_title(g, fontsize=9, fontweight='bold')
    ax.tick_params(labelsize=8)
    _, p = stats.mannwhitneyu(tumor[g].values, normal[g].values, alternative='two-sided')
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*'
    ax.text(0.5, 0.97, sig, transform=ax.transAxes, ha='center', va='top', fontsize=11, fontweight='bold')
plt.suptitle('Boxplots: Gene Expression Tumor vs Normal (*** p<0.001)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. SMOTE — balance normal samples (applied inside each CV fold)

SMOTE generates synthetic minority-class samples by interpolating between real samples and their k-nearest neighbors. It is applied **only inside each training fold** to prevent data leakage.

In [ ]:
def smote(X_min, n_synthetic, k=5, random_state=42):
    """Manual SMOTE — no external library required."""
    rng = np.random.default_rng(random_state)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X_min)
    _, indices = nn.kneighbors(X_min)
    synthetic = []
    n = len(X_min)
    for _ in range(n_synthetic):
        idx = rng.integers(0, n)
        neighbor = indices[idx, rng.integers(1, k + 1)]
        lam = rng.random()
        synthetic.append(X_min[idx] + lam * (X_min[neighbor] - X_min[idx]))
    return np.array(synthetic)

# Demonstrate on full set (for EDA only — models use within-fold SMOTE)
X_orig = combined[GENES].values
y_orig = combined['label'].values
X_min_full = X_orig[y_orig == 0]
n_needed = int(y_orig.sum()) - int((y_orig == 0).sum())
synth = smote(X_min_full, n_needed)
X_sm = np.vstack([X_orig, synth])
y_sm = np.concatenate([y_orig, np.zeros(n_needed, dtype=int)])
print(f'Original:   tumor={y_orig.sum()}  normal={(y_orig==0).sum()}')
print(f'Post-SMOTE: tumor={y_sm.sum()}  normal={(y_sm==0).sum()}')
print(f'Synthetic normal samples added: {n_needed}')

## 8. AND co-expression scoring

A sample is considered to co-express a combination if **all** genes simultaneously exceed their tumor-derived median threshold.

This enforces the biological AND requirement for bispecific/multispecific therapy: all targets must be present on the same cell.

In [ ]:
tumor_medians = {g: tumor[g].median() for g in GENES}
print('Tumor median thresholds:')
for g, m in tumor_medians.items():
    print(f'  {g:<8}: {m:.3f}')

def and_coverage(combo, df, medians):
    """Fraction of samples where ALL genes in combo exceed their tumor median."""
    mask = np.ones(len(df), dtype=bool)
    for g in combo:
        mask = mask & (df[g].values >= medians[g])
    return mask.mean()

scores_2, scores_3 = [], []
for combo in combinations(GENES, 2):
    t = and_coverage(combo, tumor,  tumor_medians)
    n = and_coverage(combo, normal, tumor_medians)
    scores_2.append({'combo': combo, 'label': ' + '.join(combo),
                     'tumor': t, 'normal': n, 'spec': t - n})
for combo in combinations(GENES, 3):
    t = and_coverage(combo, tumor,  tumor_medians)
    n = and_coverage(combo, normal, tumor_medians)
    scores_3.append({'combo': combo, 'label': ' + '.join(combo),
                     'tumor': t, 'normal': n, 'spec': t - n})

scores_2.sort(key=lambda x: (-x['spec'], -x['tumor']))
scores_3.sort(key=lambda x: (-x['spec'], -x['tumor']))

print('\nTop 10 dual targets (AND logic, ranked by specificity):')
print(f"  {'Combo':<22}  {'Tumor AND':>10}  {'Normal AND':>11}  {'Specificity':>12}")
for r in scores_2[:10]:
    print(f"  {r['label']:<22}  {r['tumor']:>10.1%}  {r['normal']:>11.1%}  {r['spec']:>+12.1%}")

print('\nTop 10 triple targets (AND logic, ranked by specificity):')
print(f"  {'Combo':<34}  {'Tumor AND':>10}  {'Normal AND':>11}  {'Specificity':>12}")
for r in scores_3[:10]:
    print(f"  {r['label']:<34}  {r['tumor']:>10.1%}  {r['normal']:>11.1%}  {r['spec']:>+12.1%}")

## 9. Coverage plot — top dual & triple targets

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, scores, title in [
    (axes[0], scores_2[:10], 'Top 10 Dual Targets (AND Logic)'),
    (axes[1], scores_3[:10], 'Top 10 Triple Targets (AND Logic)')
]:
    labels      = [r['label'] for r in scores]
    tumor_vals  = [r['tumor']  for r in scores]
    normal_vals = [r['normal'] for r in scores]
    spec_vals   = [r['spec']   for r in scores]
    x = np.arange(len(labels))
    w = 0.35
    bars_t = ax.barh(x - w/2, tumor_vals,  w, label='Tumor AND',  color='#D85A30', alpha=0.85)
    bars_n = ax.barh(x + w/2, normal_vals, w, label='Normal AND', color='#378ADD', alpha=0.85)
    ax.set_yticks(x)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Fraction with all genes above tumor median', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(axis='x', alpha=0.2)
    for i, (tv, sp) in enumerate(zip(tumor_vals, spec_vals)):
        ax.text(tv + 0.003, i - w/2, f'{tv:.1%}', va='center', fontsize=7)
        color = '#1D9E75' if sp > 0.2 else '#BA7517' if sp > 0.1 else '#888'
        ax.text(max(tumor_vals) * 1.05, i - w/2, f'{sp:+.1%}', va='center', fontsize=7, color=color, fontweight='bold')
plt.suptitle('AND Co-expression Coverage: Tumor vs Normal\n(Both/all genes must simultaneously exceed tumor median)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_and_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Model comparison — RF, DT, LR (SMOTE inside each fold)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
top2 = [tuple(r['combo']) for r in scores_2[:5]]
top3 = [tuple(r['combo']) for r in scores_3[:5]]

all_results = []
for combos, size in [(top2, '2-gene'), (top3, '3-gene')]:
    print(f'\n{size} combos:')
    for combo in combos:
        Xc = combined[list(combo)].values
        yc = combined['label'].values
        rf_f, dt_f, lr_f = [], [], []
        rf_p, dt_p, lr_p = [], [], []
        rf_r, dt_r, lr_r = [], [], []
        rf_a, dt_a, lr_a = [], [], []
        for tr, te in cv.split(Xc, yc):
            X_tr, X_te = Xc[tr], Xc[te]
            y_tr, y_te = yc[tr], yc[te]
            # SMOTE on training fold only
            X_min = X_tr[y_tr == 0]
            n_add = int(y_tr.sum()) - len(X_min)
            if n_add > 0 and len(X_min) > 5:
                s = smote(X_min, n_add, k=min(5, len(X_min) - 1))
                X_tr = np.vstack([X_tr, s])
                y_tr = np.concatenate([y_tr, np.zeros(n_add, dtype=int)])
            for name, m in [
                ('RF', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')),
                ('DT', DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced')),
                ('LR', Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=300, random_state=42, class_weight='balanced'))]))
            ]:
                m.fit(X_tr, y_tr)
                preds = m.predict(X_te)
                proba = m.predict_proba(X_te)[:, 1] if hasattr(m, 'predict_proba') else m[-1].predict_proba(m[:-1].transform(X_te))[:, 1]
                fl = [rf_f, dt_f, lr_f][['RF','DT','LR'].index(name)]
                pl = [rf_p, dt_p, lr_p][['RF','DT','LR'].index(name)]
                rl = [rf_r, dt_r, lr_r][['RF','DT','LR'].index(name)]
                al = [rf_a, dt_a, lr_a][['RF','DT','LR'].index(name)]
                fl.append(f1_score(y_te, preds))
                pl.append(precision_score(y_te, preds, zero_division=0))
                rl.append(recall_score(y_te, preds, zero_division=0))
                al.append(roc_auc_score(y_te, proba))
        label = ' + '.join(combo)
        print(f'  {label:<34}  RF F1={np.mean(rf_f):.4f} AUC={np.mean(rf_a):.4f} | DT F1={np.mean(dt_f):.4f} | LR F1={np.mean(lr_f):.4f} AUC={np.mean(lr_a):.4f}')
        all_results.append({'combo': label, 'size': size,
                            'RF_F1': np.mean(rf_f), 'RF_Prec': np.mean(rf_p), 'RF_Rec': np.mean(rf_r), 'RF_AUC': np.mean(rf_a),
                            'DT_F1': np.mean(dt_f), 'DT_Prec': np.mean(dt_p), 'DT_Rec': np.mean(dt_r), 'DT_AUC': np.mean(dt_a),
                            'LR_F1': np.mean(lr_f), 'LR_Prec': np.mean(lr_p), 'LR_Rec': np.mean(lr_r), 'LR_AUC': np.mean(lr_a)})

results_df = pd.DataFrame(all_results)
print('\nFull results table:')
print(results_df[['combo','size','RF_F1','RF_AUC','DT_F1','DT_AUC','LR_F1','LR_AUC']].to_string(index=False))

## 11. Model performance plots

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
metrics = [('RF_F1','DT_F1','LR_F1','F1'), ('RF_Prec','DT_Prec','LR_Prec','Precision'),
           ('RF_Rec','DT_Rec','LR_Rec','Recall'), ('RF_AUC','DT_AUC','LR_AUC','AUC')]
colors = {'RF':'#185FA5','DT':'#1D9E75','LR':'#D85A30'}

for ax, (rf_col, dt_col, lr_col, title) in zip(axes, metrics):
    combos = results_df['combo'].tolist()
    y = np.arange(len(combos))
    ax.barh(y - 0.25, results_df[rf_col], 0.22, color=colors['RF'], alpha=0.85, label='RF')
    ax.barh(y,        results_df[dt_col], 0.22, color=colors['DT'], alpha=0.85, label='DT')
    ax.barh(y + 0.25, results_df[lr_col], 0.22, color=colors['LR'], alpha=0.85, label='LR')
    ax.set_yticks(y)
    ax.set_yticklabels(combos, fontsize=7)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlim(0.85, 1.01)
    ax.grid(axis='x', alpha=0.2)
    ax.tick_params(labelsize=8)
    if title == 'F1':
        patches = [mpatches.Patch(color=c, label=m) for m, c in colors.items()]
        ax.legend(handles=patches, fontsize=8, loc='lower right')
    # Annotate 2-gene vs 3-gene
    for i, row in results_df.iterrows():
        color = '#D85A30' if row['size'] == '3-gene' else '#185FA5'
        ax.text(0.86, i, row['size'][0], va='center', fontsize=7, color=color, fontweight='bold')

plt.suptitle('Model Performance: RF / DT / LR (5-fold CV, SMOTE inside training folds)\n2=2-gene, 3=3-gene', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_model_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Summary table

In [ ]:
top2_label = scores_2[0]['label']
top3_label = scores_3[0]['label']

summary = pd.DataFrame([
    {'Combination': r['label'], 'Size': '2-gene',
     'Tumor AND': f"{r['tumor']:.1%}", 'Normal AND': f"{r['normal']:.1%}",
     'Specificity': f"{r['spec']:+.1%}"}
    for r in scores_2[:6]
] + [
    {'Combination': r['label'], 'Size': '3-gene',
     'Tumor AND': f"{r['tumor']:.1%}", 'Normal AND': f"{r['normal']:.1%}",
     'Specificity': f"{r['spec']:+.1%}"}
    for r in scores_3[:6]
])
print('Top AND co-expression combinations:')
print(summary.to_string(index=False))

print(f'\n{"="*60}')
print('FINAL SUMMARY')
print(f'{"="*60}')
print(f'Dataset:        TCGA Breast Invasive Carcinoma')
print(f'Tumor:          {len(tumor):,} samples')
print(f'Normal:         {len(normal):,} samples')
print(f'SMOTE added:    {n_needed:,} synthetic normal samples (within-fold)')
print(f'Co-expr logic:  AND (all genes >= tumor median simultaneously)')
print(f'\nBest dual target:   {scores_2[0]["label"]}')
print(f'  Tumor AND:    {scores_2[0]["tumor"]:.1%}')
print(f'  Normal AND:   {scores_2[0]["normal"]:.1%}')
print(f'  Specificity:  {scores_2[0]["spec"]:+.1%}')
print(f'\nBest triple target: {scores_3[0]["label"]}')
print(f'  Tumor AND:    {scores_3[0]["tumor"]:.1%}')
print(f'  Normal AND:   {scores_3[0]["normal"]:.1%}')
print(f'  Specificity:  {scores_3[0]["spec"]:+.1%}')
print(f'{"="*60}')